# FinExpert — Fine-tune Qwen3-4B for financial analysis (QLoRA, Unsloth, free Colab T4)

**Model:** `unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit` (non-thinking Instruct build, 4-bit)

**Data:** your 3 pre-split chat files — `train.jsonl` (240), `validation.jsonl` (30, used during training to pick the best epoch), `test.jsonl` (30, touched **only once** at the end for the final score).

**Steps:** GPU check → install → config → load + validate → model → train → evaluate on test → save

> Runtime → Change runtime type → **T4 GPU**, then *Run all*.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0), "|", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

## 2. Install

In [ ]:
%%capture
!pip install -q unsloth

## 3. Config (only edit this cell)

In [ ]:
MODEL_NAME     = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048          # longest example is ~2.5k characters (~1k tokens)

TRAIN_FILE = "train.jsonl"
VAL_FILE   = "validation.jsonl"
TEST_FILE  = "test.jsonl"
USE_DRIVE  = False             # True = read data from and save outputs to Google Drive
DRIVE_DIR  = "/content/drive/MyDrive/finexpert"

SEED           = 3407
LORA_R         = 16
LORA_ALPHA     = 16
EPOCHS         = 3
LEARNING_RATE  = 2e-4
BATCH_SIZE     = 2
GRAD_ACCUM     = 4             # effective batch = 8
OUTPUT_DIR     = "finexpert-qwen3-4b-lora"
EXPORT_GGUF    = False         # True = also export q4_k_m GGUF for Ollama (slow, ~10 min)

## 4. Load your 3 files

In [ ]:
import os, json

FILES = {"train": TRAIN_FILE, "validation": VAL_FILE, "test": TEST_FILE}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    paths = {k: os.path.join(DRIVE_DIR, v) for k, v in FILES.items()}
else:
    from google.colab import files
    missing = [v for v in FILES.values() if not os.path.exists(v)]
    if missing:
        print("Select ALL of these files:", missing)
        files.upload()                       # keeps the original file names
    paths = dict(FILES)

for k, p in paths.items():
    assert os.path.exists(p), f"Missing {p} - upload it or fix the name in the config cell"

def load(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            if not line.strip():
                continue
            r = json.loads(line)
            roles = [m["role"] for m in r["messages"]]
            assert roles == ["system", "user", "assistant"], f"{path} line {i}: unexpected roles {roles}"
            d = {m["role"]: m["content"].strip() for m in r["messages"]}
            out.append({
                "example_id": r["example_id"], "category": r["category"], "difficulty": r["difficulty"],
                "system": d["system"], "user": d["user"], "assistant": d["assistant"],
            })
    return out

train_rows = load(paths["train"])
val_rows   = load(paths["validation"])
test_rows  = load(paths["test"])
print("train:", len(train_rows), "| validation:", len(val_rows), "| test:", len(test_rows))

## 5. Validate the data (stops the run if anything is wrong)

Checks: empty fields, duplicate IDs, **leakage** (same prompt in two splits), junk tokens in answers, and that every task appears in every split.

In [ ]:
from collections import Counter

splits = {"train": train_rows, "validation": val_rows, "test": test_rows}
problems = []

for name, rows in splits.items():
    for r in rows:
        for k in ["example_id", "system", "user", "assistant", "category", "difficulty"]:
            if not str(r[k]).strip():
                problems.append(f"{name}/{r['example_id']}: empty {k}")
        if any(t in r["assistant"] for t in ["<think>", "</think>", "tool_call", "<|im_"]):
            problems.append(f"{name}/{r['example_id']}: junk token in answer")
    ids = Counter(r["example_id"] for r in rows)
    problems += [f"{name}: duplicate id {i}" for i, n in ids.items() if n > 1]
    cats = {r["category"] for r in rows}
    if len(cats) < 3:
        problems.append(f"{name}: only has tasks {cats}")

prompts = {n: {r["user"] for r in rows} for n, rows in splits.items()}
ids_by = {n: {r["example_id"] for r in rows} for n, rows in splits.items()}
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    if prompts[a] & prompts[b]: problems.append(f"LEAK: {len(prompts[a] & prompts[b])} identical prompts in {a} and {b}")
    if ids_by[a] & ids_by[b]:   problems.append(f"LEAK: shared example_ids in {a} and {b}")

assert not problems, "Dataset problems:\n" + "\n".join(problems[:15])

for name, rows in splits.items():
    print(f"{name:10s}", dict(Counter(r["category"] for r in rows)), dict(Counter(r["difficulty"] for r in rows)))
print("Data checks passed.")

## 6. Load model + attach LoRA

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,          # auto: fp16 on T4, bf16 on Ampere+
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)
model.print_trainable_parameters()

## 7. Format as chat text + train (loss on the assistant answer only)

In [ ]:
from datasets import Dataset

# Build ChatML by hand. tokenizer.apply_chat_template() inserted an empty
# think block into every training answer, so the model learned to start its
# replies with junk. Manual ChatML makes training text and inference prompt match exactly.
def build_prompt(r):
    return (
        "<|im_start|>system\n" + r["system"] + "<|im_end|>\n"
        + "<|im_start|>user\n" + r["user"] + "<|im_end|>\n"
        + "<|im_start|>assistant\n"
    )

def to_text(r):
    return build_prompt(r) + r["assistant"] + "<|im_end|>\n"

train_ds = Dataset.from_list([{"text": to_text(r)} for r in train_rows])
val_ds   = Dataset.from_list([{"text": to_text(r)} for r in val_rows])

lens = [len(tokenizer(t["text"]).input_ids) for t in train_ds]
print("max tokens:", max(lens), "| mean:", int(sum(lens)/len(lens)), "| limit:", MAX_SEQ_LENGTH)
assert max(lens) <= MAX_SEQ_LENGTH, "Increase MAX_SEQ_LENGTH"
assert all("<think>" not in t["text"] and "tool_call" not in t["text"] for t in train_ds), "template junk in training text"
print(train_ds[0]["text"][:700])

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# TRL renamed max_seq_length -> max_length in newer versions; support both.
_len_key = "max_length" if "max_length" in inspect.signature(SFTConfig.__init__).parameters else "max_seq_length"

args = SFTConfig(
    dataset_text_field = "text",
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs = EPOCHS,
    learning_rate = LEARNING_RATE,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    weight_decay = 0.01,
    optim = "adamw_8bit",
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 5,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 2,
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    seed = SEED,
    output_dir = "outputs",
    report_to = "none",
    packing = False,
    dataset_num_proc = 2,
    **{_len_key: MAX_SEQ_LENGTH},
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args = args,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

sample = trainer.train_dataset[0]
labels = [t for t in sample["labels"] if t != -100]
print("Trained tokens in sample 0:", len(labels), "of", len(sample["input_ids"]))
print("Loss starts at:", repr(tokenizer.decode(labels[:25])))
assert "<think>" not in tokenizer.decode(labels), "think block leaked into the training target"

In [ ]:
stats = trainer.train()
print(f"Train time: {stats.metrics['train_runtime']/60:.1f} min | peak VRAM: {torch.cuda.max_memory_reserved()/1e9:.1f} GB")

In [ ]:
import matplotlib.pyplot as plt
log = trainer.state.log_history
tr = [(l["step"], l["loss"]) for l in log if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
plt.figure(figsize=(7,4))
plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), "o-", label="val loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3); plt.show()
print("Best val loss:", trainer.state.best_metric)

## 8. Final evaluation on the held-out TEST set

The test set was never used for training or for picking the best epoch, so these are your honest numbers.

- **Label accuracy** — classification: predicted label == reference label
- **Format rate** — classification answers start with `Classification: <label>`
- **Figure fidelity** — every ₹ figure in the answer appears in the prompt or the reference (catches invented numbers)

In [ ]:
import re
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

def generate(rows_, batch_size=6, max_new_tokens=700):
    outs = []
    for i in range(0, len(rows_), batch_size):
        batch = rows_[i:i+batch_size]
        prompts = [build_prompt(r) for r in batch]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        for g in gen:
            outs.append(tokenizer.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return outs

LABEL_RE = re.compile(r"^\s*Classification:\s*(Healthy|Moderate Risk|High Risk)", re.I)
FIG_RE   = re.compile(r"₹\s*(\d+(?:\.\d+)?)\s*Cr", re.I)
figs = lambda t: {float(x) for x in FIG_RE.findall(t)}

def evaluate(rows_, preds):
    cls = [(r, p) for r, p in zip(rows_, preds) if r["category"] == "financial_classification"]
    fmt = sum(bool(LABEL_RE.match(p)) for _, p in cls)
    acc = sum(bool(LABEL_RE.match(p)) and LABEL_RE.match(p).group(1).lower() == LABEL_RE.match(r["assistant"]).group(1).lower()
              for r, p in cls)
    fid = sum(figs(p) <= (figs(r["assistant"]) | figs(r["user"])) for r, p in zip(rows_, preds))
    return {
        "classification_n": len(cls),
        "label_accuracy": round(acc / max(len(cls), 1), 3),
        "format_rate": round(fmt / max(len(cls), 1), 3),
        "figure_fidelity": round(fid / len(rows_), 3),
    }

preds = generate(test_rows)
print("empty answers:", sum(len(p) == 0 for p in preds), "| answers with junk prefix:", sum(("tool_call" in p or "<think>" in p) for p in preds))
print("FINE-TUNED:", evaluate(test_rows, preds))

In [ ]:
# Baseline (adapter switched off) on the SAME test set -> proves fine-tuning helped.
try:
    with model.disable_adapter():
        base_preds = generate(test_rows)
    print("BASE      :", evaluate(test_rows, base_preds))
except Exception as e:
    print("Baseline skipped:", e)

In [ ]:
# Which classification examples did the model get wrong?
for r, p in zip(test_rows, preds):
    if r["category"] != "financial_classification":
        continue
    ref = LABEL_RE.match(r["assistant"]).group(1)
    got = LABEL_RE.match(p).group(1) if LABEL_RE.match(p) else "NO LABEL"
    flag = "OK " if ref.lower() == got.lower() else "MISS"
    print(f'{flag} {r["example_id"]:16s} {r["difficulty"]:7s} ref={ref:13s} model={got}')

## 8b. Evaluation pipeline (repo-tracked, testable, reusable)

The scoring above lives in `src/finexpert/evaluation/` in the repo (with its
own pytest tests: `pytest tests/test_evaluation_metrics.py
tests/test_evaluation_report.py`), not only in this notebook. This cell dumps
predictions to a plain file, then calls the same evaluation code you'd run
locally any time you retrain — `python scripts/evaluate.py --test ... 
--predictions ... --out report.json` — without needing Colab or a GPU. It
also runs the stricter `numeric_mismatch` check, which catches digit-level
errors (e.g. a ratio written as `0.040` instead of `0.40`) that the
currency-only `figure_fidelity` metric can miss.

In [ ]:
# Clone/pull the repo so finexpert.evaluation is importable here, and
# so predictions/report.json can be committed alongside the dataset code.
import subprocess, os, sys, json as _json

REPO_URL = "https://github.com/mani1234765/finexpert.git"
REPO_DIR = "finexpert-repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from finexpert.evaluation.report import build_report, format_report

# 1) Dump predictions to a plain file (example_id -> prediction).
with open("predictions.jsonl", "w", encoding="utf-8") as f:
    for r, p in zip(test_rows, preds):
        f.write(_json.dumps({"example_id": r["example_id"], "prediction": p}, ensure_ascii=False) + "\n")

# 2) Score with the SAME code that's tested in the repo (not a copy of it).
predictions_by_id = {r["example_id"]: p for r, p in zip(test_rows, preds)}
report = build_report(test_rows, predictions_by_id)
print(format_report(report))

with open("report.json", "w", encoding="utf-8") as f:
    _json.dump(report, f, indent=2, ensure_ascii=False)

if USE_DRIVE:
    import shutil
    shutil.copy("predictions.jsonl", os.path.join(DRIVE_DIR, "predictions.jsonl"))
    shutil.copy("report.json", os.path.join(DRIVE_DIR, "report.json"))
else:
    from google.colab import files
    files.download("predictions.jsonl")
    files.download("report.json")

In [ ]:
for r, p in list(zip(test_rows, preds))[:3]:
    print("="*100); print(r["category"], "|", r["difficulty"], "|", r["example_id"])
    print("PROMPT   :", r["user"][-400:])
    print("REFERENCE:", r["assistant"][:500])
    print("MODEL    :", p[:500])

## 9. Save

In [ ]:
import shutil
model.save_pretrained(OUTPUT_DIR)          # LoRA adapter only (small)
tokenizer.save_pretrained(OUTPUT_DIR)

if USE_DRIVE:
    shutil.copytree(OUTPUT_DIR, os.path.join(DRIVE_DIR, OUTPUT_DIR), dirs_exist_ok=True)
    print("Saved to Drive:", os.path.join(DRIVE_DIR, OUTPUT_DIR))
else:
    from google.colab import files
    shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
    files.download(OUTPUT_DIR + ".zip")

In [ ]:
# Optional: merged GGUF for Ollama / llama.cpp (set EXPORT_GGUF=True in the config cell)
if EXPORT_GGUF:
    model.save_pretrained_gguf("finexpert-gguf", tokenizer, quantization_method="q4_k_m")
    print("Now: download finexpert-gguf/*.gguf, then `ollama create finexpert -f Modelfile`")

## Reading the results
- **val loss rising while train loss falls** → overfitting: lower `EPOCHS` to 2.
- **label_accuracy < ~0.9** → look at the MISS lines above; often a dataset labeling-rule issue, not a training issue.
- **figure_fidelity < 1.0** → the model invents numbers; inspect those rows.
- Test has only ~10 classification examples: one miss = 10 points. Treat differences of a few points as noise.
- All splits come from the same generator, so this measures *learning your format and rules*, not real-world financial skill.